# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/octane21/flyrank-internship-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window



*   one row : one content per day for one client. one row contains peformance of a content for 90 days
*   table : fact_content_daily_peformance, dim_content
*   time window : 2025-01-01 - 2026-06-30





## 2. Fields: feature / label / context / excluded


*   label : future_decline
* excluded : content_hash_id and client_hash_id are not used as model features; they are for identification, grouping, and splitting. gsc_impressions from the target window should not be included in the feature. trend_direction or trend_pct should not be used
* context : content_hash_id and client_hash_id

### Feature Definitions (Demo)

The following five features are selected for demonstration purposes to illustrate how historical Google Search Console (GSC) data can be transformed into model features and leakage.

For this example, the prediction date is April 1, 2026, and the features are calculated using historical data from Jan - March 2026. These features are available at prediction time because they are derived from data collected before the prediction date.

| Feature | Calculation | Why is it available at prediction time? |
|---|---|---|
| `impressions_90d` | Total GSC impressions during the previous 90 days. | March GSC data is already available before the April prediction date. |
| `clicks_90d` | Total GSC clicks during the previous 90 days. | March GSC data is already available before the April prediction date. |
| `ctr_90d` | Total clicks / total impressions × 100. | Calculated using historical GSC clicks and impressions. |
| `avg_position_90d` | Average GSC position during the previous 90 days. | Calculated using historical GSC position data. |
| `session_organic_90d` | Number of session during the previous 90 days. | GA4 data availability can be determined from historical records. |

**Note:** This is a demonstration of the initial feature selection process, not the final modeling feature set.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

print("Secret successfully loaded!")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN {HF_TOKEN})")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    'dim_clients':f"read_parquet('{rel}/dim_clients.parquet')",
    'dim_content':f"read_parquet('{rel}/dim_content.parquet')",
    'fact_content_daily_performance':f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')",
    'fact_content_query_90d':f"read_parquet('{rel}/fact_content_query_90d.parquet')",
}


Secret successfully loaded!


In [16]:
# Grain
dim_clients_schema = con.sql(f"""SELECT * FROM {TABLES['dim_clients']} LIMIT 1""").df().dtypes
fact_content_schema = con.sql(f"""SELECT * FROM {TABLES['fact_content_daily_performance']} LIMIT 1""").df().dtypes
date_content_client = con.sql(f"""SELECT COUNT(DISTINCT report_date) as report_date_total, COUNT(DISTINCT client_hash_id) as client_total, COUNT(DISTINCT content_hash_id) as content_total FROM {TABLES['fact_content_daily_performance']} """).df()


print("fact_content_daily_performance schema")
print(fact_content_schema,"\n")
print("dim_clients schema")
print(dim_clients_schema,"\n")
print(date_content_client.to_string(index=False),"\n")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_content_daily_performance schema
report_date                 datetime64[us]
client_hash_id                      object
content_hash_id                     object
client_has_gsc                        bool
client_has_ga4                        bool
gsc_data_available                    bool
ga4_data_available                    bool
gsc_impressions                      int64
gsc_clicks                           int64
gsc_sum_position                     int64
gsc_avg_position                   float64
ga4_pageviews                        int64
ga4_sessions                         int64
ga4_users                            int64
ga4_engaged_sessions                 int64
ga4_total_engagement_sec             int64
sessions_organic                     int64
sessions_direct                      int64
sessions_referral                    int64
sessions_social                      int64
sessions_paid                        int64
sessions_ai                          int64
ai_chatgpt      

In [17]:
# Slice's row count and date span
count_row_date_span = con.sql(f"""SELECT
                                  COUNT(DISTINCT report_date) as report_date_total,
                                  COUNT(DISTINCT client_hash_id) as client_total,
                                  COUNT(DISTINCT content_hash_id) as content_total,
                                  MIN(report_date) as start_date,
                                  MAX(report_date) as end_date
                                  FROM
                                  {TABLES['fact_content_daily_performance']}
                                 """).df()


count_row_date_span

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date_total,client_total,content_total,start_date,end_date
0,520,70,427292,2025-01-27,2026-06-30


In [18]:
# Availability
row_content_gsc_ga4 = con.sql(f"""SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT content_hash_id) AS total_content,
            COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_data_available,
            COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_data_available,
            COUNT(*) FILTER (WHERE ga4_data_available IS TRUE AND gsc_data_available IS TRUE) AS ga4_gsc_data_available,
            FROM {TABLES['fact_content_daily_performance']}""").df()

content_90_days_gsc_ga4 = con.sql(f"""
    SELECT
        COUNT(*) AS content_90_days_gsc_ga4
    FROM (
        SELECT
            content_hash_id
        FROM {TABLES['fact_content_daily_performance']}
        GROUP BY content_hash_id
        HAVING COUNT(DISTINCT report_date) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) >= 90
    ) AS content
""").df()

print(row_content_gsc_ga4.to_string(index=False))
print(content_90_days_gsc_ga4.to_string(index=False))
con.sql(f"""
    SELECT
        content_hash_id,
        MIN(report_date) as start_date_gsc_ga4,
        MAX(report_date) as end_date_gsc_ga4,
        COUNT(DISTINCT report_date) as range_gsc_ga4
    FROM (
        SELECT
            content_hash_id,
            report_date
        FROM {TABLES['fact_content_daily_performance']}
        WHERE
          gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
   )
   GROUP BY content_hash_id
   HAVING range_gsc_ga4 >= 90
   ORDER BY start_date_gsc_ga4, end_date_gsc_ga4
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  total_content  gsc_data_available  ga4_data_available  ga4_gsc_data_available
   78835655         427292            28970051             2816455                 2453375
 content_90_days_gsc_ga4
                    4619


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,start_date_gsc_ga4,end_date_gsc_ga4,range_gsc_ga4
0,content_6b4f9e61e903e486,2025-10-29,2026-04-28,99
1,content_d96a8c6c700e15c3,2025-10-29,2026-06-08,91
2,content_f3161b76ca328fd1,2025-10-29,2026-06-09,92
3,content_f0e5bb8c98cb95d7,2025-10-29,2026-06-10,114
4,content_1047d37c6f3935ba,2025-10-29,2026-06-12,161
...,...,...,...,...
4614,content_0e3f93e573464a71,2026-03-30,2026-06-30,92
4615,content_97ff27515379c0ca,2026-03-31,2026-06-29,91
4616,content_d1f8f969c7b729ec,2026-03-31,2026-06-30,92
4617,content_4d61b13ba2fff509,2026-03-31,2026-06-30,91


## Leakage trap

In [19]:
# fetch content daily data from 2026 Jan to Mar that gsc and ga4 is available and April as label

data = con.sql(f"""
  WITH content_id_hv_criteria AS (
        SELECT
            content_hash_id,
        FROM {TABLES['fact_content_daily_performance']}
        WHERE
          report_date >= DATE '2026-01-01'
          AND report_date <= '2026-03-31'
          AND gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
          GROUP BY content_hash_id
        HAVING  COUNT(DISTINCT report_date) >= 90
   )

   SELECT *
   FROM {TABLES['fact_content_daily_performance']} as fact

   INNER JOIN content_id_hv_criteria ON
   fact.content_hash_id=content_id_hv_criteria.content_hash_id
   WHERE report_date >= DATE '2026-01-01'
          AND report_date <= '2026-04-30'
""").df()

data

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,ga4_total_engagement_sec,sessions_organic,sessions_direct,sessions_referral,sessions_social,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,content_hash_id_1
0,2026-01-01,client_9958f0a7ae1df715,content_01ab236521ddf2f9,True,True,True,True,30,1,525,17.500000,15,14,14,1,540,2,0,1,0,0,0,0,0,0,0,0,0,0,8,2026-01,content_01ab236521ddf2f9
1,2026-01-04,client_9958f0a7ae1df715,content_01ab236521ddf2f9,True,True,True,True,36,2,1153,32.027778,9,9,9,2,325,4,0,0,0,0,0,0,0,0,0,0,0,0,6,2026-01,content_01ab236521ddf2f9
2,2026-01-01,client_23a62021009f63c4,content_e8a52cf3d5988c07,True,True,True,True,693,3,5964,8.606061,21,10,10,1,50,11,3,0,0,0,0,0,0,0,0,0,0,0,1,2026-01,content_e8a52cf3d5988c07
3,2026-01-01,client_23a62021009f63c4,content_249e9d28a0bb94b0,True,True,True,True,458,2,1670,3.646288,18,9,9,1,31,17,2,0,0,0,0,0,0,0,0,0,0,0,1,2026-01,content_249e9d28a0bb94b0
4,2026-01-01,client_23a62021009f63c4,content_40baa8f1016f5742,True,True,True,True,1860,6,9764,5.249462,29,15,13,0,16,23,0,0,0,0,0,0,0,0,0,0,0,0,0,2026-01,content_40baa8f1016f5742
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3235,2026-04-29,client_23a62021009f63c4,content_c4f5512e1c243ed2,True,True,True,True,925,18,7179,7.761081,40,18,14,0,2,40,0,0,0,0,0,0,0,0,0,0,0,0,0,2026-04,content_c4f5512e1c243ed2
3236,2026-04-29,client_23a62021009f63c4,content_5460a0e854602a47,True,True,True,True,208,0,5169,24.850962,5,3,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2026-04,content_5460a0e854602a47
3237,2026-04-29,client_23a62021009f63c4,content_7c95e4f6c39bb730,True,True,True,True,578,11,4626,8.003460,69,33,33,2,469,41,0,0,0,0,0,0,0,0,0,0,0,0,2,2026-04,content_7c95e4f6c39bb730
3238,2026-04-30,client_9958f0a7ae1df715,content_01ab236521ddf2f9,True,True,True,True,11,2,422,38.363636,8,7,7,1,21,4,0,0,0,0,0,0,0,0,0,0,0,0,3,2026-04,content_01ab236521ddf2f9


In [20]:
import pandas as pd
pd.set_option('display.max_columns', None)

In [21]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3240 entries, 0 to 3239
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   report_date               3240 non-null   datetime64[us]
 1   client_hash_id            3240 non-null   object        
 2   content_hash_id           3240 non-null   object        
 3   client_has_gsc            3240 non-null   bool          
 4   client_has_ga4            3240 non-null   bool          
 5   gsc_data_available        3240 non-null   bool          
 6   ga4_data_available        3240 non-null   bool          
 7   gsc_impressions           3240 non-null   int64         
 8   gsc_clicks                3240 non-null   int64         
 9   gsc_sum_position          3240 non-null   int64         
 10  gsc_avg_position          3240 non-null   float64       
 11  ga4_pageviews             3240 non-null   int64         
 12  ga4_sessions        

In [22]:
data['report_date'].dt.month

,report_date
0,1
1,1
2,1
3,1
4,1
...,...
3235,4
3236,4
3237,4
3238,4


In [23]:
features = (data[data['report_date']<='2026-03-31']
          .groupby(['content_hash_id'])
          .agg(
        impressions_90d=("gsc_impressions", "sum"),
        clicks_90d=("gsc_clicks", "sum"),
        avg_position_90d=("gsc_avg_position", "mean"),
        sessions_organic_90d=("sessions_organic", "sum"),
          ).reset_index()
            )

features['ctr_90d'] = features['clicks_90d']/features['impressions_90d'] * 100
features['avg_position_90d'].astype('int64')

,avg_position_90d
0,31
1,9
2,10
3,5
4,8
5,31
6,5
7,4
8,21
9,7


In [24]:
march_imp = (data[data['report_date'].dt.month==3]
          .groupby(['content_hash_id'])
          .agg(
        impressions_march=("gsc_impressions", "sum")
          ).reset_index()
            )

In [25]:
march_imp

,content_hash_id,impressions_march
0,content_01ab236521ddf2f9,1322
1,content_1fe2fb98dc650e7d,23023
2,content_227d9704d7286471,20900
3,content_249e9d28a0bb94b0,15977
4,content_331c22ad79508edf,14854
5,content_36e53e9c707674fc,194579
6,content_40baa8f1016f5742,59311
7,content_4c15fe8dc370f3cd,71760
8,content_5460a0e854602a47,10926
9,content_5f4599a9cff00b3e,16596


In [27]:
imp_future_30d = (
    data[data['report_date'] > '2026-03-31']
    .groupby(['content_hash_id'])
    .agg(
        impressions_future_30d=('gsc_impressions','sum')
    ).reset_index()
)

In [28]:
dataset = (
    features
    .merge(
        imp_future_30d,
        on=['content_hash_id'],
        how='inner'
    )
    .merge(
        march_imp,
        on=['content_hash_id'],
        how='inner'
    )
)

In [29]:
dataset['future_decline'] =  (((dataset['impressions_future_30d']-dataset['impressions_march'])/dataset['impressions_march']*100)< -20).astype(int)

In [30]:
dataset= dataset.drop('content_hash_id', axis=1)

## Honest Score

In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score,
    average_precision_score,
    roc_auc_score
)

X = dataset.drop(['impressions_future_30d', 'impressions_march', 'future_decline'], axis=1)
y = dataset['future_decline']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression()
model.fit(X_train, y_train)


y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
precision = precision_score(y_test, y_pred, zero_division=0)

avg_precision = average_precision_score(y_test, y_prob)

roc_auc = roc_auc_score(y_test, y_prob)


# Display results
print(f"Precision:        {precision:.4f}")
print(f"Average Precision: {avg_precision:.4f}")
print(f"ROC-AUC:          {roc_auc:.4f}")

Precision:        0.0000
Average Precision: 0.4500
ROC-AUC:          0.5000


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [32]:
scoring = X_test.copy()

scoring["actual_label"] = y_test.values

scoring["decline_probability"] = (
    model.predict_proba(X_test)[:, 1]
)
scoring["pred_label"] = (
    model.predict(X_test)
)

scoring = scoring.sort_values(
    "decline_probability",
    ascending=False
)

scoring.head()

,impressions_90d,clicks_90d,avg_position_90d,sessions_organic_90d,ctr_90d,actual_label,decline_probability,pred_label
11,42380,612,6.076361,1261,1.444077,0,0.080836,0
8,60857,293,21.750306,721,0.481457,1,0.049014,0
9,48798,879,7.904965,1808,1.801303,0,0.019913,0
13,32195,634,11.301057,1218,1.969250,0,0.016192,0
0,2879,57,31.025508,86,1.979854,1,0.003615,0


## Leakage

Giving features that form the label ('impressions_future_30d', 'impressions_march')

In [33]:
X = dataset.drop('future_decline', axis=1)
y = dataset['future_decline']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression()
model.fit(X_train, y_train)


y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
precision = precision_score(y_test, y_pred, zero_division=0)

avg_precision = average_precision_score(y_test, y_prob)

roc_auc = roc_auc_score(y_test, y_prob)

print(f"Precision:        {precision:.4f}")
print(f"Average Precision: {avg_precision:.4f}")
print(f"ROC-AUC:          {roc_auc:.4f}")

Precision:        1.0000
Average Precision: 1.0000
ROC-AUC:          1.0000


In [34]:
scoring = X_test.copy()

scoring["actual_label"] = y_test.values

scoring["decline_probability"] = (
    model.predict_proba(X_test)[:, 1]
)
scoring["pred_label"] = (
    model.predict(X_test)
)

scoring = scoring.sort_values(
    "decline_probability",
    ascending=False
)

scoring.head()

,impressions_90d,clicks_90d,avg_position_90d,sessions_organic_90d,ctr_90d,impressions_future_30d,impressions_march,actual_label,decline_probability,pred_label
8,60857,293,21.750306,721,0.481457,5479,10926,1,1.000000e+00,1
0,2879,57,31.025508,86,1.979854,590,1322,1,9.999680e-01,1
11,42380,612,6.076361,1261,1.444077,15516,14968,0,3.700585e-32,0
13,32195,634,11.301057,1218,1.969250,13522,12353,0,1.963950e-32,0
21,66427,1678,5.704672,3623,2.526081,21624,19243,0,1.551066e-47,0


In [36]:
X = dataset.drop(['future_decline', 'impressions_march'], axis=1)
y = dataset['future_decline']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression()
model.fit(X_train, y_train)


y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
precision = precision_score(y_test, y_pred, zero_division=0)

avg_precision = average_precision_score(y_test, y_prob)

roc_auc = roc_auc_score(y_test, y_prob)

print(f"Precision:        {precision:.4f}")
print(f"Average Precision: {avg_precision:.4f}")
print(f"ROC-AUC:          {roc_auc:.4f}")
scoring = X_test.copy()

scoring["actual_label"] = y_test.values

scoring["decline_probability"] = (
    model.predict_proba(X_test)[:, 1]
)
scoring["pred_label"] = (
    model.predict(X_test)
)

scoring = scoring.sort_values(
    "decline_probability",
    ascending=False
)

scoring.head()

Precision:        1.0000
Average Precision: 1.0000
ROC-AUC:          1.0000


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,impressions_90d,clicks_90d,avg_position_90d,sessions_organic_90d,ctr_90d,impressions_future_30d,actual_label,decline_probability,pred_label
8,60857,293,21.750306,721,0.481457,5479,1,1.000000e+00,1
0,2879,57,31.025508,86,1.979854,590,1,1.000000e+00,1
11,42380,612,6.076361,1261,1.444077,15516,0,2.717553e-15,0
13,32195,634,11.301057,1218,1.969250,13522,0,1.953117e-17,0
9,48798,879,7.904965,1808,1.801303,21979,0,5.309758e-30,0


In [35]:
X = dataset.drop(['future_decline','impressions_future_30d',], axis=1)
y = dataset['future_decline']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression()
model.fit(X_train, y_train)


y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
precision = precision_score(y_test, y_pred, zero_division=0)

avg_precision = average_precision_score(y_test, y_prob)

roc_auc = roc_auc_score(y_test, y_prob)

print(f"Precision:        {precision:.4f}")
print(f"Average Precision: {avg_precision:.4f}")
print(f"ROC-AUC:          {roc_auc:.4f}")
scoring = X_test.copy()

scoring["actual_label"] = y_test.values

scoring["decline_probability"] = (
    model.predict_proba(X_test)[:, 1]
)
scoring["pred_label"] = (
    model.predict(X_test)
)

scoring = scoring.sort_values(
    "decline_probability",
    ascending=False
)

scoring.head()

Precision:        0.0000
Average Precision: 0.4500
ROC-AUC:          0.5000


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,impressions_90d,clicks_90d,avg_position_90d,sessions_organic_90d,ctr_90d,impressions_march,actual_label,decline_probability,pred_label
11,42380,612,6.076361,1261,1.444077,14968,0,0.069378,0
8,60857,293,21.750306,721,0.481457,10926,1,0.063545,0
9,48798,879,7.904965,1808,1.801303,16596,0,0.018470,0
13,32195,634,11.301057,1218,1.969250,12353,0,0.014118,0
0,2879,57,31.025508,86,1.979854,1322,1,0.003833,0


## 4. Data limits

The daily performance table contains GSC and GA4 availability indicators, but data availability is not guaranteed for every client-content-date combination. This may result in incomplete historical windows and potentially insufficient observations for reliable feature and target construction.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.